# 🧠 Redes Neuronales — Notebook de Experimentación

> Este notebook está diseñado para ejecutarse en **Google Colab** o **CodeLab**. Permite experimentación interactiva con el boilerplate del proyecto.

## Instrucciones
1. Sube este cuaderno a Google Colab
2. Selecciona GPU como runtime (Menú → Runtime → Change runtime type)
3. Ejecuta las celdas en orden

## 📦 Paso 1: Instalar dependencias

> Ejecuta esta celda una sola vez al inicio.

In [ ]:
!pip install tensorflow keras numpy pandas scikit-learn matplotlib seaborn pyyaml tqdm Pillow -q

## ⚙️ Paso 2: Cargar configuración

> Modifica las variables según tu problema (imágenes, texto, etc.)

In [ ]:
import yaml

# Configurar el tipo de problema aquí:
# Opciones: 'image_classification', 'text_sentiment', 'time_series'
PROBLEM_TYPE = 'image_classification'  # ← CAMBIAR SEGÚN TU PROYECTO

config = {
    'project': {'problem_type': PROBLEM_TYPE},
    'data': {
        'raw_dir': '/content/data/raw/',
        'preprocessed_dir': '/content/data/preprocessed/',
        'train_split': 0.7,
        'validation_split': 0.15,
    },
    'model': {
        'architecture': 'cnn',  # Opciones: cnn, lstm
        'input_shape': [224, 224, 3],  # Cambiar para texto/series
        'num_classes': 2,
    },
    'training': {
        'epochs': 20,
        'batch_size': 32,
        'learning_rate': 0.001,
    }
}

# Guardar config temporal
with open('/content/config_notebook.yaml', 'w') as f:
    yaml.dump(config, f)

print('✅ Configuración cargada')
print(f'   Problema: {PROBLEM_TYPE}')
print(f'   Modelo: {config["model"]["architecture"]}')

## 🖼️ Paso 3a: Cargar dataset de imágenes

> Si trabajas con imágenes, usa este bloque. Sube tus datos a Google Drive o usa Kaggle.

In [ ]:
import os
from pathlib import Path

# Montar Google Drive (si necesitas archivos)
# from google.colab import drive
# drive.mount('/content/drive')

# Crear estructura de datos si no existe
raw_dir = Path('/content/data/raw/')
proc_dir = Path('/content/data/preprocessed/')
proc_dir.mkdir(parents=True, exist_ok=True)

# Verificar estructura esperada:
# data/raw/
#   clase_0/  (imágenes de basura orgánica, por ejemplo)
#   clase_1/  (imágenes de basura inorgánica)

print(f'📂 Directorio raw: {raw_dir}')
if raw_dir.exists():
    classes = [d.name for d in raw_dir.iterdir() if d.is_dir()]
    print(f'   Clases detectadas: {classes}')
else:
    print('   ⚠️ data/raw/ no existe. Sube tus imágenes aquí.')

print(f'\n📂 Directorio preprocessed: {proc_dir}')

## 🏗️ Paso 4: Construir modelo

In [ ]:
import tensorflow as tf

# Construir CNN según configuración
input_shape = tuple(config['model']['input_shape'])
num_classes = config['model']['num_classes']
lr = config['training']['learning_rate']

print('🏗️ Construyendo modelo CNN...')

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'✅ Modelo construido: {sum(model.count_params()):,} parámetros')
model.summary()

## 🚀 Paso 5: Entrenar modelo

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    '/content/models/best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Entrenar (ajustar batch_size según memoria de Colab)
batch_size = 32
epochs = config['training']['epochs']

print(f'🚀 Entrenando: {epochs} epochs, batch size={batch_size}')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

## 📊 Paso 6: Evaluar y visualizar resultados

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
if 'val_accuracy' in history.history:
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
if 'val_loss' in history.history:
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Clasificación en test set
from sklearn.metrics import classification_report

predictions = model.predict(X_test)
y_pred = predictions.argmax(axis=1)

print(classification_report(y_test, y_pred))

## 📥 Paso 7: Descargar modelo entrenado

In [ ]:
# Descargar el modelo a tu computadora
from google.colab import files

files.download('/content/models/best_model.keras')